# Economics Real-Data Results / Economics 真实数据结果

This notebook consolidates the formal economics real-data suite: CMDL, plain-LSTM baseline, and the three core ablation variants.
本 notebook 汇总 economics 真实数据正式实验：CMDL、plain-LSTM baseline，以及三个核心 ablation 变体。

## Sections / 结构

- Setup: repo paths, imports, and the single configuration cell.
- Optional direct-run: programmatic training hooks for CMDL, baseline, and ablation.
- Unified comparison: build normalized run-level tables for forecasting and lag/proxy diagnostics.
- Multi-seed summaries and plots: aggregate seeds 0/1/2 and export figures.
- Prediction snapshots: compare canonical runs on a few entities.

## Defaults / 默认行为

- The active plan is `formal_target`.
- Direct-run switches are opt-in and default to `False`.
- The notebook reads from `data/economics/processed/economics_cleaned_long.csv`.
- Outputs are organized under `outputs/notebook_economics/<plan>/...`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "config").exists() and (path / "data").exists() and (path / "experiments").exists()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError(f"Could not find repo root from {current}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"repo_root = {repo_root}")

In [ ]:
from argparse import Namespace
import importlib

from evaluation import economics_comparison as economics_comparison_module
from experiments import run_economics as run_economics_module
from experiments import run_economics_ablation as run_economics_ablation_module
from experiments import run_economics_lstm_baseline as run_economics_lstm_baseline_module

economics_comparison_module = importlib.reload(economics_comparison_module)
run_economics_module = importlib.reload(run_economics_module)
run_economics_ablation_module = importlib.reload(run_economics_ablation_module)
run_economics_lstm_baseline_module = importlib.reload(run_economics_lstm_baseline_module)

build_economics_comparison = economics_comparison_module.build_economics_comparison
build_task_table = economics_comparison_module.build_task_table
build_interpretability_table = economics_comparison_module.build_interpretability_table
run_cmdl_experiment = run_economics_module.run_experiment
run_baseline_experiment = run_economics_lstm_baseline_module.run_experiment
run_ablation_suite = run_economics_ablation_module.run_suite

PRESET_CONFIGS = {
    "quick_check": {"epochs": 3, "patience": 1, "log_every": 1},
    "notebook_medium": {"epochs": 30, "patience": 8, "log_every": 5},
    "formal_target": {"epochs": 120, "patience": 20, "log_every": 10},
}

ACTIVE_PLAN = "formal_target"
SEEDS = [0, 1, 2]
CANONICAL_SEED = SEEDS[0]
RUN_CMDL = False
RUN_BASELINE = False
RUN_ABLATIONS = False
DISABLE_MLFLOW = False
TARGET_COLUMN = "ctfp"

DATA_PATH = repo_root / "data" / "economics" / "processed" / "economics_cleaned_long.csv"
OUTPUT_ROOT = repo_root / "outputs" / "notebook_economics" / ACTIVE_PLAN
CMDL_OUTPUT_DIR = OUTPUT_ROOT / "cmdl"
BASELINE_OUTPUT_DIR = OUTPUT_ROOT / "plain_lstm"
ABLATION_OUTPUT_DIR = OUTPUT_ROOT / "ablation"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
PLOTS_DIR = OUTPUT_ROOT / "comparison_plots"

for path in [OUTPUT_ROOT, CMDL_OUTPUT_DIR, BASELINE_OUTPUT_DIR, ABLATION_OUTPUT_DIR, COMPARISON_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

COMMON_ARGS = {
    "csv_path": str(DATA_PATH),
    "year_start": 1980,
    "year_end": 2023,
    "train_end_year": 2007,
    "val_end_year": 2013,
    "target_column": TARGET_COLUMN,
    "max_missing_share": 0.15,
    "lr": 1e-3,
    "lambda_r": 0.1,
    "temperature": 1.0,
    "lag_bias_strength": 1.0,
    "grad_clip": 1.0,
    "device": "auto",
    "disable_mlflow": DISABLE_MLFLOW,
}

plan_table = pd.DataFrame.from_dict(PRESET_CONFIGS, orient="index")
notebook_switches = pd.DataFrame(
    {
        "value": [
            ACTIVE_PLAN,
            DATA_PATH,
            CMDL_OUTPUT_DIR,
            BASELINE_OUTPUT_DIR,
            ABLATION_OUTPUT_DIR,
            COMPARISON_DIR,
            PLOTS_DIR,
            ", ".join(str(seed) for seed in SEEDS),
            CANONICAL_SEED,
            RUN_CMDL,
            RUN_BASELINE,
            RUN_ABLATIONS,
            DISABLE_MLFLOW,
        ]
    },
    index=[
        "ACTIVE_PLAN",
        "DATA_PATH",
        "CMDL_OUTPUT_DIR",
        "BASELINE_OUTPUT_DIR",
        "ABLATION_OUTPUT_DIR",
        "COMPARISON_DIR",
        "PLOTS_DIR",
        "SEEDS",
        "CANONICAL_SEED",
        "RUN_CMDL",
        "RUN_BASELINE",
        "RUN_ABLATIONS",
        "DISABLE_MLFLOW",
    ],
)

display(plan_table)
display(pd.Series({**COMMON_ARGS, **PRESET_CONFIGS[ACTIVE_PLAN]}, name="value").to_frame())
display(notebook_switches)
print(f"ACTIVE_PLAN = {ACTIVE_PLAN}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")

In [ ]:
cmdl_summaries: list[dict[str, object]] = []
baseline_summaries: list[dict[str, object]] = []
ablation_summary_frame = pd.DataFrame()
ablation_aggregated_frame = pd.DataFrame()

if RUN_CMDL:
    for seed in SEEDS:
        cmdl_args = Namespace(
            **COMMON_ARGS,
            **PRESET_CONFIGS[ACTIVE_PLAN],
            seed=seed,
            output_dir=str(CMDL_OUTPUT_DIR),
            experiment_name=f"E4_economics_cmdl_seed{seed}",
            smoke=ACTIVE_PLAN == "quick_check",
        )
        cmdl_summary = run_cmdl_experiment(cmdl_args)
        cmdl_summaries.append(cmdl_summary)
        print(
            f"CMDL seed={seed} test_r2={cmdl_summary['metrics']['test']['r2']:.4f} "
            f"test_mae={cmdl_summary['metrics']['test']['mae']:.4f}"
        )
else:
    print("RUN_CMDL = False; skipping direct CMDL training.")

if RUN_BASELINE:
    for seed in SEEDS:
        baseline_args = Namespace(
            **COMMON_ARGS,
            **PRESET_CONFIGS[ACTIVE_PLAN],
            seed=seed,
            output_dir=str(BASELINE_OUTPUT_DIR),
            experiment_name=f"E4_economics_lstm_seed{seed}",
            smoke=ACTIVE_PLAN == "quick_check",
        )
        baseline_summary = run_baseline_experiment(baseline_args)
        baseline_summaries.append(baseline_summary)
        print(
            f"Baseline seed={seed} test_r2={baseline_summary['metrics']['test']['r2']:.4f} "
            f"test_mae={baseline_summary['metrics']['test']['mae']:.4f}"
        )
else:
    print("RUN_BASELINE = False; skipping direct baseline training.")

if RUN_ABLATIONS:
    ablation_args = Namespace(
        **COMMON_ARGS,
        **PRESET_CONFIGS[ACTIVE_PLAN],
        variant="all",
        seeds=SEEDS,
        output_dir=str(ABLATION_OUTPUT_DIR),
        experiment_prefix="E4_economics_ablation",
        smoke=ACTIVE_PLAN == "quick_check",
    )
    ablation_summary_frame, ablation_aggregated_frame = run_ablation_suite(ablation_args)
    display(ablation_summary_frame)
    if not ablation_aggregated_frame.empty:
        display(ablation_aggregated_frame)
else:
    print("RUN_ABLATIONS = False; skipping direct ablation training.")

In [ ]:
comparison = build_economics_comparison(
    cmdl_root=CMDL_OUTPUT_DIR,
    baseline_root=BASELINE_OUTPUT_DIR,
    ablation_root=ABLATION_OUTPUT_DIR,
)
task_table = build_task_table(comparison)
interpretability_table = build_interpretability_table(comparison)

comparison_path = COMPARISON_DIR / "economics_comparison.csv"
task_table_path = COMPARISON_DIR / "forecast_comparison.csv"
interpretability_path = COMPARISON_DIR / "lag_proxy_diagnostics.csv"

comparison.to_csv(comparison_path, index=False)
task_table.to_csv(task_table_path, index=False)
interpretability_table.to_csv(interpretability_path, index=False)

print(f"comparison rows = {len(comparison)}")
print(f"task table path = {task_table_path}")
print(f"interpretability path = {interpretability_path}")
display(task_table)
display(interpretability_table)

In [ ]:
def aggregate_seed_table(frame: pd.DataFrame, metric_columns: list[str]) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=["display_name", "family", "variant"])

    available_metrics = [column for column in metric_columns if column in frame.columns]
    if not available_metrics:
        return pd.DataFrame(columns=["display_name", "family", "variant"])

    aggregated = frame.groupby(["display_name", "family", "variant"], dropna=False)[available_metrics].agg(["mean", "std"])
    aggregated.columns = [
        column if isinstance(column, str) else "_".join(part for part in column if part)
        for column in aggregated.columns.to_flat_index()
    ]
    return aggregated.reset_index().sort_values(["family", "display_name"], na_position="last").reset_index(drop=True)


task_seed_summary = aggregate_seed_table(
    comparison,
    ["test_r2", "test_mae", "test_mse", "best_val_task_loss"],
)
interpretability_seed_summary = aggregate_seed_table(
    comparison,
    [
        "test_effective_kstar_proxy_spearman_rho",
        "test_effective_kstar_mean",
        "test_effective_kstar_std",
        "test_effective_lag_entropy_mean",
        "test_proxy_signal_r2",
    ],
)

task_seed_summary.to_csv(COMPARISON_DIR / "forecast_comparison_seed_summary.csv", index=False)
interpretability_seed_summary.to_csv(COMPARISON_DIR / "lag_proxy_diagnostics_seed_summary.csv", index=False)

display(task_seed_summary)
display(interpretability_seed_summary)

In [ ]:
if task_seed_summary.empty or interpretability_seed_summary.empty:
    print("No comparison results found yet; skipping plots.")
else:
    forecast_plot_frame = task_seed_summary.copy()
    diagnostics_plot_frame = interpretability_seed_summary.copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(forecast_plot_frame["display_name"], forecast_plot_frame["test_r2_mean"], color="#1f77b4")
    axes[0].set_title("Forecast R2 by model")
    axes[0].set_ylabel("mean test R2")
    axes[0].tick_params(axis="x", rotation=25)

    axes[1].bar(forecast_plot_frame["display_name"], forecast_plot_frame["test_mae_mean"], color="#ff7f0e")
    axes[1].set_title("Forecast MAE by model")
    axes[1].set_ylabel("mean test MAE")
    axes[1].tick_params(axis="x", rotation=25)

    fig.tight_layout()
    forecast_plot_path = PLOTS_DIR / "forecast_metrics.png"
    fig.savefig(forecast_plot_path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(
        diagnostics_plot_frame["display_name"],
        diagnostics_plot_frame["test_effective_kstar_proxy_spearman_rho_mean"],
        color="#2ca02c",
    )
    axes[0].set_title("Lag-proxy alignment")
    axes[0].set_ylabel("mean test effective k* proxy rho")
    axes[0].tick_params(axis="x", rotation=25)

    axes[1].bar(
        diagnostics_plot_frame["display_name"],
        diagnostics_plot_frame["test_effective_lag_entropy_mean_mean"],
        color="#d62728",
    )
    axes[1].set_title("Lag entropy")
    axes[1].set_ylabel("mean test lag entropy")
    axes[1].tick_params(axis="x", rotation=25)

    fig.tight_layout()
    diagnostics_plot_path = PLOTS_DIR / "lag_proxy_diagnostics.png"
    fig.savefig(diagnostics_plot_path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

    print(f"Saved plots to {forecast_plot_path} and {diagnostics_plot_path}")

In [ ]:
prediction_specs = {
    "CMDL": CMDL_OUTPUT_DIR / f"E4_economics_cmdl_seed{CANONICAL_SEED}" / "predictions.csv",
    "Plain LSTM": BASELINE_OUTPUT_DIR / f"E4_economics_lstm_seed{CANONICAL_SEED}" / "predictions.csv",
    "No AC Encoder": ABLATION_OUTPUT_DIR / f"E4_economics_ablation_no_ac_encoder_seed{CANONICAL_SEED}" / "predictions.csv",
}

prediction_frames: dict[str, pd.DataFrame] = {}
for label, path in prediction_specs.items():
    if path.exists():
        prediction_frames[label] = pd.read_csv(path)

if not prediction_frames:
    print("No prediction snapshots found yet; skipping time-series comparison.")
else:
    reference_frame = next(iter(prediction_frames.values()))
    entity_codes = sorted(reference_frame["entity_code"].unique().tolist())[:2]
    fig, axes = plt.subplots(len(entity_codes), 1, figsize=(12, 4 * len(entity_codes)), sharex=True)
    if len(entity_codes) == 1:
        axes = [axes]

    for axis, entity_code in zip(axes, entity_codes):
        baseline_truth = reference_frame[reference_frame["entity_code"] == entity_code]
        axis.plot(baseline_truth["year"], baseline_truth["y_true"], color="black", linewidth=2.0, label="y_true")
        for label, frame in prediction_frames.items():
            entity_frame = frame[frame["entity_code"] == entity_code]
            axis.plot(entity_frame["year"], entity_frame["y_pred"], linewidth=1.8, label=label)
        axis.set_title(f"Entity {entity_code}: prediction snapshot")
        axis.set_ylabel(TARGET_COLUMN)
        axis.grid(alpha=0.25)
        axis.legend()

    axes[-1].set_xlabel("year")
    fig.tight_layout()
    prediction_plot_path = PLOTS_DIR / "prediction_snapshots.png"
    fig.savefig(prediction_plot_path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved prediction snapshot plot to {prediction_plot_path}")